# Chapter 5: ERG-Specific Filtering

This notebook demonstrates the complete ERG filter pipeline:
1. Median filter (spike removal)
2. Notch filter (50/60 Hz mains interference)
3. Butterworth bandpass filter (0.3-300 Hz ISCEV standard)

All filters use zero-phase implementation (`sosfiltfilt`) to preserve implicit times.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, sosfiltfilt, sosfreqz, iirnotch, tf2sos, medfilt, welch

# Set plotting style
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

## 1. Butterworth Bandpass Filter (ISCEV Standard)

The Butterworth filter provides a maximally flat passband - no ripples - which preserves ERG amplitude measurements without distortion.

In [ ]:
# Parameters
FS_HZ = 1000
LOWCUT_HZ = 0.3      # High-pass cutoff (preserves PhNR)
HIGHCUT_HZ = 300     # Low-pass cutoff (preserves OPs)
ORDER = 4            # 4th-order Butterworth (ISCEV standard)

def design_bandpass(lowcut, highcut, fs, order):
    nyquist = fs / 2
    low = lowcut / nyquist
    high = highcut / nyquist
    return butter(order, [low, high], btype='bandpass', output='sos')

def plot_filter_response(sos, fs, lowcut, highcut):
    w, h = sosfreqz(sos, worN=4096, fs=fs)
    fig, ax = plt.subplots()
    ax.semilogx(w, 20 * np.log10(np.abs(h) + 1e-12), color='#2E75B6', lw=1.8)
    ax.axvline(lowcut, color='red', ls='--', lw=1.0, label=f'High-pass: {lowcut} Hz')
    ax.axvline(highcut, color='green', ls='--', lw=1.0, label=f'Low-pass: {highcut} Hz')
    ax.axhline(-3, color='gray', ls=':', lw=0.8, label='-3 dB')
    ax.set_xlim(0.05, fs/2)
    ax.set_ylim(-80, 5)
    ax.set_xscale('log')
    ax.set_xlabel('Frequency (Hz)')
    ax.set_ylabel('Magnitude (dB)')
    ax.set_title('4th-Order Butterworth Bandpass Filter (0.3-300 Hz)')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

# Design and plot
sos_band = design_bandpass(LOWCUT_HZ, HIGHCUT_HZ, FS_HZ, ORDER)
plot_filter_response(sos_band, FS_HZ, LOWCUT_HZ, HIGHCUT_HZ)

## 2. Notch Filter (50 Hz Power-Line Interference)

The notch filter removes mains hum while leaving adjacent frequencies (49-51 Hz) within 1 dB of unity.

In [ ]:
NOTCH_HZ = 50.0
QUALITY_FACTOR = 30.0

def design_notch(notch_hz, q, fs):
    b, a = iirnotch(notch_hz, q, fs=fs)
    return tf2sos(b, a)

def plot_notch_response(sos, fs, notch_hz):
    w, h = sosfreqz(sos, worN=5000, fs=fs)
    mag_db = 20 * np.log10(np.abs(h) + 1e-12)
    fig, ax = plt.subplots()
    ax.plot(w, mag_db, color='#2E75B6', lw=1.8)
    ax.axvline(notch_hz, color='red', ls='--', lw=1.0)
    ax.axhline(-3, color='gray', ls=':', lw=0.8)
    ax.set_xlim(notch_hz - 15, notch_hz + 15)
    ax.set_ylim(-60, 3)
    ax.set_xlabel('Frequency (Hz)')
    ax.set_ylabel('Magnitude (dB)')
    ax.set_title(f'Notch Filter: {notch_hz} Hz, Q = {QUALITY_FACTOR}')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

sos_notch = design_notch(NOTCH_HZ, QUALITY_FACTOR, FS_HZ)
plot_notch_response(sos_notch, FS_HZ, NOTCH_HZ)

## 3. Median Filter (Spike Removal)

The median filter removes isolated high-amplitude spikes without distorting the underlying ERG waveform.

In [ ]:
def apply_median(signal, kernel=5):
    if kernel % 2 == 0:
        kernel += 1
    return medfilt(signal, kernel_size=kernel)

# Create test signal with spike
t = np.arange(0, 0.25, 1/FS_HZ)
clean = np.sin(2 * np.pi * 10 * t)
noisy = clean.copy()
spike_idx = int(0.08 * FS_HZ)
noisy[spike_idx] += 3.0

filtered = apply_median(noisy, kernel=5)

fig, axes = plt.subplots(2, 1, sharex=True)
axes[0].plot(t, noisy, color='red', lw=0.8)
axes[0].set_ylabel('µV')
axes[0].set_title('Before Median Filter: Spike Present')
axes[1].plot(t, filtered, color='#2E75B6', lw=0.8)
axes[1].set_xlabel('Time (ms)')
axes[1].set_ylabel('µV')
axes[1].set_title('After Median Filter: Spike Removed')
axes[0].grid(True, alpha=0.3)
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Complete Filter Pipeline

Apply all three filters in the correct order: Median → Notch → Bandpass

In [ ]:
def detect_mains(signal, fs, notch_hz=50.0, threshold=6.0):
    freqs, psd = welch(signal, fs=fs, nperseg=min(1024, len(signal)))
    idx = np.argmin(np.abs(freqs - notch_hz))
    p_notch = 10 * np.log10(psd[idx] + 1e-30)
    left = max(0, idx - 5)
    right = min(len(psd), idx + 6)
    shoulder = np.concatenate([psd[left:idx], psd[idx+1:right]])
    p_shoulder = 10 * np.log10(np.mean(shoulder) + 1e-30)
    return (p_notch - p_shoulder) > threshold

def full_pipeline(signal, fs, notch_hz=50.0):
    log = []
    sig = signal.copy()
    
    # Step 1: Median
    sig = apply_median(sig, kernel=5)
    log.append('Median filter applied')
    
    # Step 2: Notch (conditional)
    if detect_mains(sig, fs, notch_hz):
        sos_notch = design_notch(notch_hz, 30.0, fs)
        sig = sosfiltfilt(sos_notch, sig)
        log.append(f'Notch filter applied at {notch_hz} Hz')
    else:
        log.append('Notch filter skipped: no mains detected')
    
    # Step 3: Bandpass
    sos_band = design_bandpass(0.3, 300, fs, 4)
    sig = sosfiltfilt(sos_band, sig)
    log.append('Butterworth bandpass applied (0.3-300 Hz)')
    
    return sig, log

# Test on synthetic signal
t = np.arange(0, 0.25, 1/FS_HZ)
synthetic = np.sin(2 * np.pi * 10 * t) + 0.3 * np.sin(2 * np.pi * 50 * t) + 0.1 * np.random.randn(len(t))

filtered_signal, filter_log = full_pipeline(synthetic, FS_HZ, notch_hz=50.0)

print("Filter Pipeline Log:")
for step in filter_log:
    print(f"  - {step}")

fig, axes = plt.subplots(2, 1, sharex=True)
axes[0].plot(t, synthetic, color='red', lw=0.8)
axes[0].set_ylabel('µV')
axes[0].set_title('Raw Signal (with noise)')
axes[1].plot(t, filtered_signal, color='#2E75B6', lw=0.8)
axes[1].set_xlabel('Time (ms)')
axes[1].set_ylabel('µV')
axes[1].set_title('Filtered Signal')
axes[0].grid(True, alpha=0.3)
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Summary

- The Butterworth bandpass filter provides a maximally flat passband (0.3-300 Hz) with 24 dB/octave roll-off
- The notch filter removes 50/60 Hz mains interference with minimal effect on adjacent frequencies
- The median filter removes isolated spikes without distorting the ERG waveform
- All filters use zero-phase implementation (`sosfiltfilt`) to preserve implicit times
- The complete pipeline applies filters in the correct order: Median → Notch → Bandpass